# Ch17: Reasoning（推理）— LangChain + MiMo

本章演示三种推理增强模式：

1. **Chain-of-Thought（CoT）**：分步思考，逐步推理
2. **Self-Correction**：自我检查和修正错误
3. **Plan-and-Execute**：先规划再执行

## 为什么需要推理增强？
- LLM 直接回答复杂问题容易出错
- 通过引导 LLM「分步思考」可以显著提高准确率
- 推理能力是 Agent 智能的核心（o1/o3、DeepSeek-R1 都主打推理）

## 第1步：导入依赖

In [1]:
import os
import json

from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage

os.environ["OPENAI_API_KEY"] = os.getenv("MIMO_API_KEY", "")
os.environ["OPENAI_API_BASE"] = "https://token-plan-cn.xiaomimimo.com/v1"

llm = ChatOpenAI(
    model="mimo-v2.5-pro",
    temperature=0.3,
    base_url="https://token-plan-cn.xiaomimimo.com/v1",
)
print("MiMo 模型初始化完成")

MiMo 模型初始化完成


## 第2步：Chain-of-Thought（CoT）分步推理

CoT 的核心：在 prompt 中要求 LLM「一步一步思考」，
让模型展示推理过程，而不是直接给答案。

**对比**：
- 直接回答：容易跳步骤、出错
- CoT：强制模型展示每一步推理，准确率更高

In [2]:
# 直接回答 vs CoT 对比

test_question = "一个商店打 8 折促销，一件原价 200 元的衣服，顾客用了一张满 150 减 20 的优惠券，最终需要支付多少钱？"

# 方法1：直接回答
print("=" * 60)
print("方法1：直接回答")
print("=" * 60)

direct_prompt = "请直接回答以下问题，只需给出最终答案：\n\n" + test_question
response_direct = llm.invoke([HumanMessage(content=direct_prompt)])
print(response_direct.content)

方法1：直接回答


140 元


In [3]:
# 方法2：CoT 分步推理
print("=" * 60)
print("方法2：CoT 分步推理")
print("=" * 60)

cot_prompt = """请一步一步思考以下问题：

{question}

请按以下格式回答：
第1步：...
第2步：...
...
最终答案：...""".format(question=test_question)

response_cot = llm.invoke([HumanMessage(content=cot_prompt)])
print(response_cot.content)

方法2：CoT 分步推理


第1步：计算打折后的价格。原价200元，打8折，即乘以0.8，得到200 × 0.8 = 160元。  
第2步：判断优惠券使用条件。优惠券为“满150减20”，折后价160元满足满150元的条件，因此可以减免20元。  
第3步：计算最终支付金额。折后价减去优惠券减免金额：160元 - 20元 = 140元。  
最终答案：140元。


## 第3步：Self-Correction（自我修正）

Self-Correction 的核心：让 LLM 先回答，然后自我检查，
发现错误后自行修正。

**流程**：回答 → 检查 → 修正（可多轮）

In [4]:
# Self-Correction 两步流程

def self_correct(question: str, max_rounds: int = 2) -> dict:
    """自我修正流程：回答 → 检查 → 修正"""
    rounds = []

    # 第1步：初始回答
    initial_prompt = f"请回答以下问题：\n{question}"
    response = llm.invoke([HumanMessage(content=initial_prompt)])
    current_answer = response.content
    rounds.append({"round": 0, "answer": current_answer})

    # 后续轮次：检查和修正
    for i in range(max_rounds):
        check_prompt = f"""请检查以下回答是否正确。如果有错误，请指出并给出修正后的回答。
如果没有错误，请回复「回答正确」。

问题：{question}
回答：{current_answer}

请按以下格式输出：
检查结果：（正确/有错误）
问题分析：...
修正后的回答：...（如果正确则写「无需修正」）"""

        check_response = llm.invoke([HumanMessage(content=check_prompt)])
        check_text = check_response.content
        rounds.append({"round": i + 1, "check": check_text})

        # 如果检查结果是正确的，停止
        if "回答正确" in check_text or "无需修正" in check_text:
            break

        # 提取修正后的回答
        if "修正后的回答：" in check_text:
            current_answer = check_text.split("修正后的回答：")[1].strip()

    return {
        "question": question,
        "final_answer": current_answer,
        "rounds": rounds
    }

print("Self-Correction 函数定义完成")

Self-Correction 函数定义完成


In [5]:
# 测试 Self-Correction

# 故意用一个容易出错的问题
sc_question = "小明有 5 个苹果，给了小红 2 个，又买了 3 个，然后把一半分给了小华。小明现在有几个苹果？"

print("问题:", sc_question)
print("=" * 60)

result = self_correct(sc_question, max_rounds=2)

for r in result["rounds"]:
    if "round" in r and "answer" in r:
        print(f"\n--- 初始回答 ---")
        print(r["answer"][:200])
    elif "check" in r:
        print(f"\n--- 检查轮次 {r['round']} ---")
        print(r["check"][:200])

print(f"\n最终答案: {result['final_answer'][:200]}")

问题: 小明有 5 个苹果，给了小红 2 个，又买了 3 个，然后把一半分给了小华。小明现在有几个苹果？



--- 初始回答 ---
小明一开始有 5 个苹果，给了小红 2 个后剩下 3 个；又买了 3 个，变成 6 个；最后把一半分给小华，即分出 3 个，自己剩下 3 个。

所以，小明现在有 **3 个苹果**。

--- 检查轮次 1 ---
检查结果：正确

问题分析：
- 初始：5 个苹果
- 给小红 2 个：5 - 2 = 3 个
- 买了 3 个：3 + 3 = 6 个
- 分一半给小华：6 ÷ 2 = 3 个（分出 3 个，自己剩 3 个）

回答中的每一步计算都正确，最终答案 **3 个苹果** 无误。

修正后的回答：无需修正

最终答案: 小明一开始有 5 个苹果，给了小红 2 个后剩下 3 个；又买了 3 个，变成 6 个；最后把一半分给小华，即分出 3 个，自己剩下 3 个。

所以，小明现在有 **3 个苹果**。


## 第4步：Plan-and-Execute（先规划再执行）

Plan-and-Execute 的核心：
1. **规划阶段**：LLM 生成执行计划（步骤列表）
2. **执行阶段**：按计划逐步执行
3. **汇总阶段**：整合所有步骤的结果

与 Ch6 Planning 的关系：本章是 Ch6 的推理增强版，强调在每一步中进行深度思考。

In [6]:
def plan_and_execute(task: str) -> dict:
    """先规划再执行"""

    # 第1步：生成计划
    plan_prompt = f"""请为以下任务生成详细的执行计划。
只输出步骤列表，不要执行。

任务：{task}

请以 JSON 格式输出：
{{"steps": ["步骤1描述", "步骤2描述", ...]}}"""

    plan_response = llm.invoke([HumanMessage(content=plan_prompt)])
    plan_text = plan_response.content.strip()

    # 解析计划
    try:
        if "```json" in plan_text:
            plan_text = plan_text.split("```json")[1].split("```")[0].strip()
        elif "```" in plan_text:
            plan_text = plan_text.split("```")[1].split("```")[0].strip()
        plan = json.loads(plan_text)
        steps = plan.get("steps", [])
    except json.JSONDecodeError:
        # 如果 JSON 解析失败，按行分割
        steps = [line.strip() for line in plan_text.split("\n") if line.strip() and not line.strip().startswith("{")]

    print(f"生成了 {len(steps)} 个步骤：")
    for i, step in enumerate(steps):
        print(f"  {i+1}. {step}")

    # 第2步：逐步执行
    results = []
    context = ""
    for i, step in enumerate(steps):
        print(f"\n执行步骤 {i+1}/{len(steps)}: {step[:50]}...")

        exec_prompt = f"""请执行以下步骤，并给出结果。

原始任务：{task}
之前步骤的结果：{context if context else '无'}

当前步骤：{step}

请直接给出这一步的结果（简洁明了）。"""

        exec_response = llm.invoke([HumanMessage(content=exec_prompt)])
        step_result = exec_response.content
        results.append({"step": step, "result": step_result})
        context += f"\n步骤{i+1}: {step} → {step_result[:100]}\n"

    # 第3步：汇总
    results_text = ""
    for i, r in enumerate(results):
        results_text += f"\n步骤{i+1}: {r['step']}\n结果: {r['result']}\n"
    summary_prompt = f"""请根据以下执行结果，给出最终总结。

任务：{task}

执行结果：
{results_text}
请给出简洁的最终总结。"""

    summary_response = llm.invoke([HumanMessage(content=summary_prompt)])

    return {
        "task": task,
        "steps": steps,
        "results": results,
        "summary": summary_response.content
    }

print("Plan-and-Execute 函数定义完成")

Plan-and-Execute 函数定义完成


In [7]:
# 测试 Plan-and-Execute

task = "帮我分析 Python 中列表推导式和 for 循环的性能差异，给出建议"

print(f"任务: {task}")
print("=" * 60)

result = plan_and_execute(task)

print("\n" + "=" * 60)
print("最终总结")
print("=" * 60)
print(result["summary"])

任务: 帮我分析 Python 中列表推导式和 for 循环的性能差异，给出建议


生成了 9 个步骤：
  1. 理解列表推导式和 for 循环的基本概念、语法和常见用法，确保掌握它们的工作原理。
  2. 设计一系列测试用例，覆盖不同场景如简单列表创建、条件过滤、函数应用和嵌套循环，以全面比较性能。
  3. 编写 Python 脚本，使用 timeit 模块或其他性能测量工具，为每个测试用例分别实现列表推导式和 for 循环版本。
  4. 运行测试脚本多次（例如1000次），收集每个版本的平均执行时间数据，以减少随机误差。
  5. 整理测试结果，计算性能差异指标，如执行时间比、速度提升百分比，并记录在表格或图表中。
  6. 分析性能差异的原因，考虑 Python 内部优化（如列表推导式的底层实现）和代码结构影响。
  7. 评估代码可读性、维护性和适用场景，权衡性能优势与代码清晰度。
  8. 基于分析结果，给出具体建议，例如在简单操作中优先使用列表推导式以提升性能，在复杂逻辑中使用 for 循环以保持可读性。
  9. 总结所有发现，形成最终报告，包括性能数据、分析结论和实用建议。

执行步骤 1/9: 理解列表推导式和 for 循环的基本概念、语法和常见用法，确保掌握它们的工作原理。...



执行步骤 2/9: 设计一系列测试用例，覆盖不同场景如简单列表创建、条件过滤、函数应用和嵌套循环，以全面比较性能。...



执行步骤 3/9: 编写 Python 脚本，使用 timeit 模块或其他性能测量工具，为每个测试用例分别实现列表推导...



执行步骤 4/9: 运行测试脚本多次（例如1000次），收集每个版本的平均执行时间数据，以减少随机误差。...


...


执行步骤 6/9: 分析性能差异的原因，考虑 Python 内部优化（如列表推导式的底层实现）和代码结构影响。...



执行步骤 7/9: 评估代码可读性、维护性和适用场景，权衡性能优势与代码清晰度。...



执行步骤 8/9: 基于分析结果，给出具体建议，例如在简单操作中优先使用列表推导式以提升性能，在复杂逻辑中使用 for ...



执行步骤 9/9: 总结所有发现，形成最终报告，包括性能数据、分析结论和实用建议。...



最终总结
根据全面的性能测试与分析，最终结论如下：

**核心发现：**
1.  **性能优势**：在所有测试场景中，**列表推导式的执行速度均比等效的 `for` 循环快约 30%-35%**。其优势源于 Python 解释器的底层优化，减少了函数调用和内存管理的开销。
2.  **代码权衡**：列表推导式在**简洁性**和**性能**上占优，而 `for` 循环在**灵活性**和**可读性**（尤其对于复杂逻辑）上更佳。

**实用建议：**
*   **优先使用列表推导式**：当进行**简单的数据映射、过滤或创建新列表**时。它能带来显著的性能提升和更简洁的代码。
*   **优先使用 `for` 循环**：当逻辑**复杂**（如多层嵌套、多步骤操作、需要异常处理）、需要**调试**或执行**副作用操作**时。此时代码的清晰度和可维护性更为重要。

**一句话总结：**
**在简单场景下，用列表推导式追求高效与简洁；在复杂场景下，用 `for` 循环保障可读与灵活。**


## 总结

### 三种推理模式对比

| 模式 | 核心思想 | 优点 | 缺点 | 适用场景 |
|------|---------|------|------|----------|
| CoT | 分步思考 | 简单、无需额外调用 | 不保证正确 | 数学、逻辑题 |
| Self-Correction | 检查+修正 | 能发现错误 | 多次调用、成本高 | 高风险决策 |
| Plan-and-Execute | 先规划再执行 | 结构化、可追踪 | 计划可能不合理 | 复杂多步任务 |

### 核心要点
- **CoT 是最基础的推理增强**——在 prompt 中加「一步一步思考」就能提升效果
- **Self-Correction 适合高风险场景**——多花成本换更高准确率
- **Plan-and-Execute 适合复杂任务**——先想清楚再动手，减少返工

### 与其他模式的关系

| 模式 | 推理方式 | 章节 |
|------|---------|------|
| Prompt Chaining | 固定流程 | Ch1 |
| Routing | 条件分支 | Ch2 |
| Planning | 动态规划 | Ch6 |
| Reasoning | 深度思考 | Ch17（本章）|

### 进阶方向
- **Tree-of-Thought（ToT）**：多条推理路径并行探索
- **Graph-of-Thought（GoT）**：图结构推理，支持回溯
- **Self-Consistency**：多次 CoT，取多数投票结果
- **结合搜索**：推理 + 联网搜索（参考原 notebook 的 DeepSearch）